In [4]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import random_split
import joblib
import json
from pathlib import Path
import copy

#BASE_DIR = Path(__file__).resolve().parent
import sys
#sys.path.append("/work/home/acychkqsp6/zrm/MasterThesis/tools")
sys.path.append("../../tools")
import architec as acht
import evaluation as eva
import prepare as prep

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [5]:
import importlib
#importlib.reload(eva)


In [6]:
NumSpecies = 20
featgam = NumSpecies**2 - NumSpecies
task_num = 3+NumSpecies # 0: num. of stable states, 1: stable-state entropy, 2: diversity, 3: abundance, 3+NumSpecies: switching species, 3+2*NumSpecies: oscillation

# The folder storing datasets splited latter.
#folder_path = "/work/home/acychkqsp6/zrm/MasterThesis/Databases/Ver1.9/ver1.9.6_upto20"
#folder_path = "../../Databases/Ver1.9/ver1.9.6_upto20"
folder_path = "../../Databases/Ver1.9/ver1.9.6_upto20_v2"

for rep in range(1,4):
    #split_df = pd.read_csv(f"/work/home/acychkqsp6/zrm/MasterThesis/Principle_design_SynCom/Fig2/split_MD_v1/split_MD_{rep}.csv")
    #split_df = pd.read_csv(f"split_MD_v1/split_MD_{rep}.csv")
    split_df = pd.read_csv(f"../Fig2/split_ver1.9.6_upto20_v2/split_{rep}.csv")

    #Trainset
    narray_tr = prep.load_split_dataset(
        split_df,
        folder_path,
        richness_list=[10],
        #richness_list=list(range(3,21)),
        subset="tr"
    )

    #Testset
    narray_te = prep.load_split_dataset(
        split_df,
        folder_path,
        richness_list=[10],
        #richness_list=list(range(3,21)),
        subset="te"
    )

    #-------------------------------------------------------
    Xg_tr, XA_tr, Xd_tr, y_tr, scaler_g, scaler_d, idx_tr = prep.prepare_train(
        narray_tr,
        NumSpecies,
        featgam,
        task_num,
        #sample_size=97000,
        random_state=42
    )

    Xg_te, XA_te, Xd_te, y_te, idx_te = prep.prepare_test(
        narray_te,
        NumSpecies,
        featgam,
        scaler_g,
        scaler_d,
        task_num,
        #sample_size=20000,
        random_state=42
    )

    train_dataset = acht.CommunityDataset_MultiLabel(Xg_tr, XA_tr, Xd_tr, y_tr)
    test_dataset = acht.CommunityDataset_MultiLabel(Xg_te, XA_te, Xd_te, y_te)

    n_val = int(len(train_dataset)*0.2)
    n_train = len(train_dataset)-n_val
    train_ds, val_ds = random_split(train_dataset, [n_train,n_val])

    train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=512, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

    model = acht.MLP(NumSpecies, featgam, output_dim=NumSpecies).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    early_stop = acht.EarlyStopping(patience=8, min_delta=1e-4)
    history = {"loss":[], "val_loss":[]}

    best_state = None

    for epoch in range(140):
        model.train()
        train_loss = 0
        
        for Xg,XA,Xd,y in train_loader:

            Xg = Xg.to(device)
            XA = XA.to(device)
            Xd = Xd.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(Xg,XA,Xd)

            loss = criterion(logits,y)
            loss.backward()

            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        model.eval()

        val_loss = 0

        with torch.no_grad():

            for Xg,XA,Xd,y in val_loader:

                Xg = Xg.to(device)
                XA = XA.to(device)
                Xd = Xd.to(device)
                y = y.to(device)

                logits = model(Xg,XA,Xd)
                loss = criterion(logits,y)
                val_loss += loss.item()

        val_loss /= len(val_loader)

        history["loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"Epoch {epoch+1} "
            f"train={train_loss:.5f} "
            f"val={val_loss:.5f}"
        )

        if val_loss < early_stop.best_loss:
            #best_state = model.state_dict()
            best_state = copy.deepcopy(model.state_dict())

        if early_stop.step(val_loss):
            print("Early stopping")
            break

    model.load_state_dict(best_state)

    # Model saving
    #save_dir = Path(f"/work/home/acychkqsp6/zrm/MasterThesis/saved_models/swi_species_15fold/sp3to20_SS_transpt3_r{rep}")
    #save_dir = Path(f"../../saved_models/swi_species_15fold/sp3to20_SS_transpt3_r{rep}")
    save_dir = Path(f"../../saved_models/swi_species_3fold/sp10_SS_MLP_r{rep}")

    save_dir.mkdir(parents=True, exist_ok=True)  # auto-create directory
    torch.save(model.state_dict(), save_dir / "model.pt")
    joblib.dump(scaler_g, save_dir / "scaler_g.pkl")
    joblib.dump(scaler_d, save_dir / "scaler_d.pkl")

    config = {
        "model_type": "MLP",
        "NumSpecies": NumSpecies,
        "featgam": featgam,
        "task_num": task_num,
        "hidden_dims": [512, 256, 128],
        "dropout": 0.1,
        "output_dim": NumSpecies
    }

    with open(save_dir / "config.json", "w") as f:
        json.dump(config, f)

    model.eval()
    preds = []

    with torch.no_grad():

        for Xg, XA, Xd, _ in test_loader:

            Xg = Xg.to(device)
            XA = XA.to(device)
            Xd = Xd.to(device)

            logits = model(Xg, XA, Xd)
            prob = torch.sigmoid(logits)

            preds.append(prob.cpu().numpy())

    y_prob = np.concatenate(preds, axis=0)

    threshold = 0.3
    metrics_list = []

    rich = 10
    for i in range(rich):

        metrics_i = eva.eval_binary(y_te[:, i], y_prob[:, i], threshold=threshold)
        metrics_i["Species"] = i
        metrics_list.append(metrics_i)

    df_metrics_species = pd.DataFrame(metrics_list)

    metrics = df_metrics_species.drop(columns=["Species"]).mean().to_dict()
    metrics["test_richness"] = rich

    display(pd.DataFrame([metrics]))


tr shape = (78594, 465)
te shape = (19648, 465)
Epoch 1 train=0.32963 val=0.28383
Epoch 2 train=0.27140 val=0.26215
Epoch 3 train=0.26014 val=0.25852
Epoch 4 train=0.25566 val=0.25768
Epoch 5 train=0.25109 val=0.25836
Epoch 6 train=0.24766 val=0.26100
Epoch 7 train=0.24502 val=0.25718
Epoch 8 train=0.24154 val=0.25481
Epoch 9 train=0.23876 val=0.25581
Epoch 10 train=0.23552 val=0.25759
Epoch 11 train=0.23324 val=0.25725
Epoch 12 train=0.23022 val=0.25836
Epoch 13 train=0.22808 val=0.26007
Epoch 14 train=0.22533 val=0.26206
Epoch 15 train=0.22369 val=0.26005
Epoch 16 train=0.22013 val=0.26217
Early stopping


,Accuracy,Balanced_Acc,F1,MCC,AUC,AUPRC,AUPRC_rel,Brier,pos_rate,test_richness
0,0.677611,0.703663,0.592299,0.375188,0.776504,0.592325,0.948064,0.169441,0.304087,10


tr shape = (78594, 465)
te shape = (19648, 465)
Epoch 1 train=0.33493 val=0.28872
Epoch 2 train=0.27761 val=0.26651
Epoch 3 train=0.26262 val=0.26035
Epoch 4 train=0.25691 val=0.25893
Epoch 5 train=0.25282 val=0.25634
Epoch 6 train=0.24909 val=0.25687
Epoch 7 train=0.24610 val=0.25827
Epoch 8 train=0.24296 val=0.25689
Epoch 9 train=0.24042 val=0.25798
Epoch 10 train=0.23761 val=0.25866
Epoch 11 train=0.23470 val=0.25962
Epoch 12 train=0.23269 val=0.25786
Epoch 13 train=0.23030 val=0.25879
Early stopping


,Accuracy,Balanced_Acc,F1,MCC,AUC,AUPRC,AUPRC_rel,Brier,pos_rate,test_richness
0,0.671397,0.700516,0.590668,0.369986,0.773228,0.584548,0.912768,0.171162,0.305604,10


tr shape = (78594, 465)
te shape = (19648, 465)
Epoch 1 train=0.33377 val=0.28281
Epoch 2 train=0.27465 val=0.26246
Epoch 3 train=0.26148 val=0.25656
Epoch 4 train=0.25552 val=0.25760
Epoch 5 train=0.25186 val=0.25546
Epoch 6 train=0.24845 val=0.25359
Epoch 7 train=0.24548 val=0.25451
Epoch 8 train=0.24255 val=0.25543
Epoch 9 train=0.24003 val=0.25516
Epoch 10 train=0.23653 val=0.25556
Epoch 11 train=0.23467 val=0.25596
Epoch 12 train=0.23157 val=0.25717
Epoch 13 train=0.22944 val=0.25645
Epoch 14 train=0.22635 val=0.25635
Early stopping


,Accuracy,Balanced_Acc,F1,MCC,AUC,AUPRC,AUPRC_rel,Brier,pos_rate,test_richness
0,0.66874,0.696241,0.58733,0.363056,0.767728,0.575186,0.873284,0.173528,0.307054,10
